# Fake News Detection - Exploratory Data Analysis

This notebook explores the sample dataset used to train the AI Fake News Detector.
We examine class balance, text length distributions, vocabulary patterns, and linguistic
signals that differentiate fake from real news articles.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

warnings.filterwarnings('ignore')

# Make project root importable
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / 'data' / 'sample' / 'sample_news.csv'

# Notebook-wide style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

print(f'Project root : {PROJECT_ROOT}')
print(f'Data file    : {DATA_PATH}')
print(f'File exists  : {DATA_PATH.exists()}')

In [ ]:
# ---------------------------------------------------------------------------
# Load sample data
# ---------------------------------------------------------------------------
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    # Fallback: generate a small synthetic dataset so the notebook runs anywhere
    import random, string
    random.seed(42)
    np.random.seed(42)

    fake_templates = [
        'BREAKING: {adj} politician CAUGHT in massive {noun} scandal!!!',
        'You won\'t BELIEVE what {adj} officials are hiding about {noun}!',
        'SHOCKING: {adj} government secretly planning {noun} takeover!!',
        'EXCLUSIVE: {adj} celebrity reveals {noun} conspiracy to public!!!',
        'ALERT: {adj} agency caught lying about {noun} for decades!!!',
    ]
    real_templates = [
        'Study finds {adj} approach reduces {noun} by 15 percent.',
        'Officials announce new {adj} policy on {noun} management.',
        'Research confirms {adj} link between diet and {noun} health.',
        'Local {adj} council votes to improve {noun} infrastructure.',
        'Scientists publish {adj} findings on {noun} climate trends.',
    ]
    adjectives = ['major', 'local', 'global', 'annual', 'recent',
                  'federal', 'independent', 'national', 'regional', 'new']
    nouns      = ['economy', 'health', 'education', 'climate', 'security',
                  'technology', 'housing', 'energy', 'water', 'transport']

    rows = []
    for _ in range(1000):
        tmpl = random.choice(fake_templates)
        text = tmpl.format(adj=random.choice(adjectives), noun=random.choice(nouns))
        rows.append({'text': text, 'label': 'FAKE'})
    for _ in range(1000):
        tmpl = random.choice(real_templates)
        text = tmpl.format(adj=random.choice(adjectives), noun=random.choice(nouns))
        rows.append({'text': text, 'label': 'REAL'})

    df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
    print('Using synthetic fallback dataset.')

print(f'Shape  : {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# ---------------------------------------------------------------------------
# Basic statistics
# ---------------------------------------------------------------------------
print('=== Label distribution ===')
print(df['label'].value_counts())
print()
print('=== Class balance (%) ===')
print(df['label'].value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')
print()

# Derive numeric features for describe()
df['text_length']  = df['text'].str.len()
df['word_count']   = df['text'].str.split().str.len()
df['caps_ratio']   = df['text'].apply(lambda t: sum(1 for c in t if c.isupper()) / max(len(t), 1))
df['exclaim_count']= df['text'].str.count('!')

print('=== Descriptive statistics (all) ===')
display(df[['text_length', 'word_count', 'caps_ratio', 'exclaim_count']].describe().round(3))

print('\n=== Descriptive statistics by label ===')
display(df.groupby('label')[['text_length', 'word_count', 'caps_ratio', 'exclaim_count']].describe().round(3))

In [ ]:
# ---------------------------------------------------------------------------
# Text length distribution — Fake vs Real
# ---------------------------------------------------------------------------
fig = px.histogram(
    df,
    x='text_length',
    color='label',
    nbins=50,
    barmode='overlay',
    opacity=0.7,
    color_discrete_map={'FAKE': '#EF553B', 'REAL': '#00CC96'},
    labels={'text_length': 'Article Length (characters)', 'count': 'Count'},
    title='Text Length Distribution: Fake vs Real'
)
fig.update_layout(legend_title_text='Label')
fig.show()

In [ ]:
# ---------------------------------------------------------------------------
# Word count boxplot by label
# ---------------------------------------------------------------------------
fig, ax = plt.subplots()
sns.boxplot(
    data=df,
    x='label',
    y='word_count',
    palette={'FAKE': '#EF553B', 'REAL': '#00CC96'},
    width=0.45,
    ax=ax
)
ax.set_title('Word Count Distribution by Label', fontsize=14, fontweight='bold')
ax.set_xlabel('Label')
ax.set_ylabel('Word Count')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Top 20 most common words — Fake vs Real
# ---------------------------------------------------------------------------
import re
from collections import Counter

STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'is', 'are', 'was', 'were', 'be', 'been',
    'has', 'have', 'had', 'that', 'this', 'it', 'its', 'as', 'not', 'no',
    'about', 'new', 'will', 'can', 'his', 'her', 'their', 'our', 'your',
}

def top_words(texts, n=20):
    tokens = re.findall(r'\b[a-z]{3,}\b', ' '.join(texts).lower())
    tokens = [t for t in tokens if t not in STOPWORDS]
    return Counter(tokens).most_common(n)

fake_top = top_words(df.loc[df['label'] == 'FAKE', 'text'])
real_top = top_words(df.loc[df['label'] == 'REAL', 'text'])

fake_df = pd.DataFrame(fake_top, columns=['word', 'count']).assign(label='FAKE')
real_df = pd.DataFrame(real_top, columns=['word', 'count']).assign(label='REAL')
combined = pd.concat([fake_df, real_df], ignore_index=True)

fig = px.bar(
    combined,
    x='count',
    y='word',
    color='label',
    facet_col='label',
    orientation='h',
    color_discrete_map={'FAKE': '#EF553B', 'REAL': '#00CC96'},
    title='Top 20 Most Common Words — Fake vs Real',
    labels={'count': 'Frequency', 'word': ''}
)
fig.update_yaxes(matches=None, showticklabels=True)
fig.update_layout(showlegend=False, height=520)
fig.show()

In [ ]:
# ---------------------------------------------------------------------------
# Caps ratio comparison — Fake vs Real
# ---------------------------------------------------------------------------
fig = px.violin(
    df,
    x='label',
    y='caps_ratio',
    color='label',
    box=True,
    points='outliers',
    color_discrete_map={'FAKE': '#EF553B', 'REAL': '#00CC96'},
    title='Caps Ratio (uppercase chars / total chars) by Label',
    labels={'caps_ratio': 'Caps Ratio', 'label': 'Label'}
)
fig.update_layout(showlegend=False)
fig.show()

# Summary stats
print(df.groupby('label')['caps_ratio'].agg(['mean', 'median', 'std']).round(4))

In [ ]:
# ---------------------------------------------------------------------------
# Exclamation mark count distribution
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, sharey=True)

for ax, label, color in zip(axes, ['FAKE', 'REAL'], ['#EF553B', '#00CC96']):
    subset = df.loc[df['label'] == label, 'exclaim_count']
    counts = subset.value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{label} Articles', fontweight='bold')
    ax.set_xlabel('Number of "!" per Article')
    ax.set_ylabel('Article Count')
    mean_val = subset.mean()
    ax.axvline(mean_val, color='black', linestyle='--', linewidth=1.2,
               label=f'Mean = {mean_val:.2f}')
    ax.legend(fontsize=9)

plt.suptitle('Exclamation Mark Count Distribution: Fake vs Real',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Mean exclamation marks per article:')
print(df.groupby('label')['exclaim_count'].mean().round(3))

## Conclusions

### Key findings from the EDA

**1. Class balance**  
The dataset is approximately balanced between FAKE and REAL articles (~50/50), which means
accuracy is a fair primary metric and no class-weighting adjustments are strictly necessary.

**2. Text length**  
Real articles tend to be moderately longer and more consistently structured. Fake articles
show higher variance in length — many are very short, headline-style clickbait pieces.

**3. Word count**  
The boxplot confirms that fake articles cluster at lower word counts with more outliers,
while real articles have a tighter, higher distribution — suggesting real journalism uses
more detailed prose.

**4. Vocabulary differences**  
Fake articles heavily use emotionally charged, sensational words ("breaking", "shocking",
"caught", "massive", "secret"). Real articles favour neutral, factual terms ("study",
"research", "policy", "officials", "confirms"). This vocabulary gap is the primary signal
exploited by the TF-IDF features.

**5. Caps ratio**  
Fake articles use significantly more uppercase characters. All-caps words are a common
misinformation tactic to create urgency. This makes `caps_ratio` a strong standalone
linguistic feature.

**6. Exclamation marks**  
Fake articles contain far more exclamation marks on average. Real news articles rarely use
them. This single character-level feature carries meaningful discriminative power.

### Modelling implications

- TF-IDF features (word + character n-grams) should capture vocabulary and style differences effectively.
- Linguistic features (`caps_ratio`, `exclaim_count`, `word_count`) provide complementary,
  model-agnostic signals.
- The ensemble approach (LR + SVM + RF) is well-suited: LR/SVM will leverage the sparse
  TF-IDF space while RF naturally handles the denser linguistic feature set.